<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-04-image-and-multimodal-generation/lesson-4.2-stable-diffusion-ecosystem/notebooks/exercises-4_2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 4.2: Stable Diffusion Ecosystem

*Module 4 · Image, Vision & Multimodal AI*

> Understand every component inside Stable Diffusion, generate images with Python in 5 lines, then learn ControlNet, LoRA, and ComfyUI — the tools professionals actually use.

---

## About this notebook

This notebook contains the **10 runnable code examples** from the published lesson `lesson-4.2.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — The Architecture — 3 Components Working Together — `part1_explore_components.py`
2. Step 2 — Text-to-Image — Generate in 5 Lines — `part2a_basic_generation.py`
3. Step 2 — Text-to-Image — Generate in 5 Lines — `part2b_full_control.py`
4. Step 2 — Text-to-Image — Generate in 5 Lines — `part2c_batch_compare.py`
5. Step 3 — Image-to-Image — Transform Existing Images — `part3_img2img.py`
6. Step 4 — ControlNet — Precise Control Over Generation — `part4_controlnet.py`
7. Step 5 — LoRA — Teach New Styles With Tiny Fine-Tunes — `part5_lora_use.py`
8. Step 5 — LoRA — Teach New Styles With Tiny Fine-Tunes — `part5_lora_train.py`
9. Step 6 — ComfyUI — The Professional Workflow Tool — `part6_comfyui_api.py`
10. Step 8 — Inference Optimization — 10× Faster Generation — `optimization.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q transformers torch numpy matplotlib requests pillow opencv-python accelerate


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · The Architecture — 3 Components Working Together

Stable Diffusion isn't one model. It's three models collaborating.

**`part1_explore_components.py`** · _Block 1 of 10_

EXPLORE: See each component individually — pip install diffusers transformers accelerate torch


In [ ]:
# =============================================
# EXPLORE: See each component individually
# pip install diffusers transformers accelerate torch
# =============================================

from diffusers import StableDiffusionPipeline
import torch

# Load the full model (downloads ~5 GB first time)
pipe = StableDiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-2-1",
    torch_dtype=torch.float16,  # half precision = faster + less memory
)

# ── Peek at the 3 components ──

# Component 1: Text Encoder (CLIP)
text_encoder = pipe.text_encoder
print("TEXT ENCODER (CLIP):")
print(f"  Parameters: {sum(p.numel() for p in text_encoder.parameters()):,}")
print(f"  Output: 77 tokens × 1024 dimensions\n")

# Component 2: U-Net (Denoiser)
unet = pipe.unet
print("U-NET (Denoiser):")
print(f"  Parameters: {sum(p.numel() for p in unet.parameters()):,}")
print(f"  This is the BIG one — ~860M parameters\n")

# Component 3: VAE (Compressor/Decompressor)
vae = pipe.vae
print("VAE (Autoencoder):")
print(f"  Parameters: {sum(p.numel() for p in vae.parameters()):,}")
print(f"  Compresses 512×512 → 64×64 (64× less data!)\n")

# Scheduler (noise schedule from Lesson 4.1)
scheduler = pipe.scheduler
print(f"SCHEDULER: {scheduler.__class__.__name__}")
print(f"  Timesteps: {scheduler.config.num_train_timesteps}")

print("""
Total model size: ~1.2 billion parameters
  - CLIP text encoder: ~340M (understands text)
  - U-Net denoiser:    ~860M (creates the image)  
  - VAE:               ~84M  (compresses/decompresses)
""")


> **What just happened?** We loaded Stable Diffusion and inspected its 3 components. The U-Net is the biggest (~860M parameters) — it does the heavy lifting of denoising. CLIP (~340M) understands your text. The VAE (~84M) handles compression. Together: ~1.2 billion parameters. We also saw the scheduler — the noise schedule from Lesson 4.1 that controls the denoising pace. Now let's use them to generate images.


### Step 2 · Text-to-Image — Generate in 5 Lines

From text prompt to photorealistic image. The simplest way first, then we'll go deeper.

**`part2a_basic_generation.py`** · _Block 2 of 10_

TEXT-TO-IMAGE: 5 lines to generate an image


In [ ]:
# =============================================
# TEXT-TO-IMAGE: 5 lines to generate an image
# =============================================

from diffusers import StableDiffusionPipeline
import torch

# Load model (use SDXL for best quality)
pipe = StableDiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-2-1",
    torch_dtype=torch.float16,
).to("cuda")  # move to GPU

# Generate!
image = pipe("A beautiful sunset over the Taj Mahal, golden hour photography").images[0]
image.save("taj_sunset.png")
print("Image saved!")

# That's it. 5 lines: import, load, generate, save, done.


**`part2b_full_control.py`** · _Block 3 of 10_

FULL CONTROL: Every parameter explained


In [ ]:
# =============================================
# FULL CONTROL: Every parameter explained
# =============================================

from diffusers import StableDiffusionPipeline
import torch

pipe = StableDiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-2-1",
    torch_dtype=torch.float16,
).to("cuda")

prompt = "A majestic Bengal tiger in a misty Indian forest, photorealistic, 8k, detailed fur"

# Negative prompt: what you DON'T want in the image
negative = "blurry, low quality, cartoon, anime, deformed, ugly, extra limbs"

image = pipe(
    prompt=prompt,
    negative_prompt=negative,         # things to avoid
    num_inference_steps=30,           # denoising steps (20-50, higher = better but slower)
    guidance_scale=7.5,               # how closely to follow the prompt (5-15)
    height=512,                       # image height in pixels
    width=512,                        # image width in pixels
    generator=torch.Generator("cuda").manual_seed(42),  # reproducible results
).images[0]

image.save("tiger_forest.png")

print("""
Parameters explained:

  num_inference_steps (20-50):
    • 20 = fast but rough
    • 30 = good balance (recommended)
    • 50 = high quality but slow
    
  guidance_scale (1-20):
    • 1   = ignores your prompt (random art)
    • 7.5 = good balance (recommended)  
    • 15  = follows prompt very strictly (can be over-saturated)
    • 20+ = usually looks bad (too constrained)
    
  negative_prompt:
    • Lists things you DON'T want
    • "blurry, low quality" almost always improves results
    • Specific negatives for your use case (e.g., "text, watermark")
    
  seed:
    • Same seed + same prompt = SAME image every time
    • Change seed = completely different image
    • Great for reproducing results and A/B testing prompts
""")


> **What just happened?** We generated a photorealistic tiger image with full control over every parameter. The key knobs: steps (quality vs speed), guidance_scale (how tightly to follow the prompt), negative_prompt (what to avoid), and seed (reproducibility). These are the same parameters available in Midjourney and DALL-E — now you understand what they actually do inside the model.


#### Batch generation and prompt comparison

**`part2c_batch_compare.py`** · _Block 4 of 10_

COMPARE: Same scene, different styles


In [ ]:
# =============================================
# COMPARE: Same scene, different styles
# =============================================

import matplotlib.pyplot as plt

prompts = [
    "A chai stall in Mumbai, watercolor painting style",
    "A chai stall in Mumbai, cyberpunk neon style, futuristic",
    "A chai stall in Mumbai, Studio Ghibli anime style",
    "A chai stall in Mumbai, photorealistic, 35mm film, golden hour",
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
seed = torch.Generator("cuda").manual_seed(42)

for i, prompt in enumerate(prompts):
    img = pipe(
        prompt, negative_prompt="blurry, low quality",
        num_inference_steps=25, guidance_scale=7.5,
        generator=torch.Generator("cuda").manual_seed(42),
    ).images[0]
    
    axes[i].imshow(img)
    style = prompt.split(",")[1].strip()[:25]
    axes[i].set_title(style, fontsize=10)
    axes[i].axis("off")

plt.suptitle("Same scene, 4 art styles", fontsize=13)
plt.tight_layout()
plt.savefig("style_comparison.png", dpi=100)
plt.show()


### Step 3 · Image-to-Image — Transform Existing Images

Don't start from noise. Start from an existing image and modify it with AI.

**`part3_img2img.py`** · _Block 5 of 10_

IMAGE-TO-IMAGE: Transform an existing image — Instead of starting from pure noise, start


In [ ]:
# =============================================
# IMAGE-TO-IMAGE: Transform an existing image
# Instead of starting from pure noise, start
# from an existing image (partially noised).
# =============================================

from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image
import torch

# Load img2img pipeline
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-2-1",
    torch_dtype=torch.float16,
).to("cuda")

# Load your starting image
init_image = Image.open("your_photo.jpg").resize((512, 512))

# Transform it!
result = pipe(
    prompt="A beautiful oil painting of the same scene, impressionist style, vivid colors",
    image=init_image,                # your starting image
    strength=0.6,                    # how much to change (0.0 = keep original, 1.0 = ignore original)
    guidance_scale=7.5,
    num_inference_steps=30,
).images[0]

result.save("transformed.png")

print("""
The 'strength' parameter is KEY:

  strength = 0.2 → keeps 80% of original (subtle style change)
  strength = 0.5 → half original, half AI (good for style transfer)
  strength = 0.7 → mostly AI-generated (keeps composition, changes style)
  strength = 1.0 → completely regenerates (ignores original image)

Try different strengths on the same image to find the sweet spot!
""")

# ── Multiple transformations of the same image ──
transforms = [
    ("Same scene at night, moonlight, dark sky", 0.6),
    ("Same scene in winter, snow everywhere, cold atmosphere", 0.5),
    ("Same scene as a pencil sketch, black and white", 0.7),
    ("Same scene in pixel art style, retro game aesthetic", 0.7),
]

for prompt, strength in transforms:
    img = pipe(prompt=prompt, image=init_image, strength=strength,
              num_inference_steps=25, guidance_scale=7.5).images[0]
    filename = prompt.split(",")[0].replace(" ", "_")[:30] + ".png"
    img.save(filename)
    print(f"Saved: {filename}")


> **What just happened?** Instead of starting from pure noise, we started from an existing image. The strength parameter controls how much noise to add to the original before denoising: 0.2 = barely change it, 0.7 = heavy transformation. The AI keeps the overall structure/composition from the original but applies the new style described in the prompt. This is how AI-powered photo editors work.


### Step 4 · ControlNet — Precise Control Over Generation

Tell the AI EXACTLY where to put things using edges, poses, or depth maps.

**`part4_controlnet.py`** · _Block 6 of 10_

CONTROLNET: Generate images that match a — specific structure (edges, pose, depth).


In [ ]:
# =============================================
# CONTROLNET: Generate images that match a 
# specific structure (edges, pose, depth).
# =============================================

from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from diffusers.utils import load_image
import torch
import cv2
import numpy as np
from PIL import Image

# ── Step 1: Extract edges from a reference image ──
def get_canny_edges(image_path, low=100, high=200):
    """Extract edge map from any image using Canny detector."""
    img = cv2.imread(image_path)
    img = cv2.resize(img, (512, 512))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, low, high)
    return Image.fromarray(edges)

# Get edges from a reference photo
edge_image = get_canny_edges("reference_photo.jpg")
edge_image.save("edges.png")
print("Edge map extracted!")

# ── Step 2: Load ControlNet + Stable Diffusion ──
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny",
    torch_dtype=torch.float16,
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-2-1",
    controlnet=controlnet,
    torch_dtype=torch.float16,
).to("cuda")

# ── Step 3: Generate images that follow the edges ──
styles = [
    "A photorealistic building, architecture photography",
    "A fantasy castle, digital art, dramatic lighting",
    "A cyberpunk city, neon lights, rain, blade runner style",
]

for style in styles:
    result = pipe(
        prompt=style,
        image=edge_image,             # ControlNet follows these edges
        num_inference_steps=25,
        guidance_scale=7.5,
        controlnet_conditioning_scale=0.8,  # how strictly to follow edges (0-1)
    ).images[0]
    
    filename = style.split(",")[0].replace(" ", "_")[:20] + ".png"
    result.save(filename)
    print(f"Generated: {filename}")

print("""
All 3 images follow the SAME edges (same building shape)
but with completely different styles!

This is why ControlNet is essential for production work:
  - Product photos: keep the product shape, change the background
  - Architecture: keep the building, change the style
  - Fashion: keep the pose, change the outfit
  - Interiors: keep the room layout, change the decor
""")


> **What just happened?** We extracted edges from a photo, then generated 3 completely different images that all follow the same edge structure. A photorealistic building, a fantasy castle, and a cyberpunk city — all with the same shape. The controlnet_conditioning_scale controls how strictly to follow the edges (0.8 = closely, 0.4 = loosely). This is how professionals maintain specific compositions while changing styles.


### Step 5 · LoRA — Teach New Styles With Tiny Fine-Tunes

Fine-tune Stable Diffusion on 20 images of your face, product, or art style. In 30 minutes.

**`part5_lora_use.py`** · _Block 7 of 10_

USING LoRAs: Load pre-trained style adapters — Thousands of free LoRAs on CivitAI and HF


In [ ]:
# =============================================
# USING LoRAs: Load pre-trained style adapters
# Thousands of free LoRAs on CivitAI and HF
# =============================================

from diffusers import StableDiffusionPipeline
import torch

pipe = StableDiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-2-1",
    torch_dtype=torch.float16,
).to("cuda")

# ── Load a LoRA from Hugging Face ──
# Example: a pixel art style LoRA
pipe.load_lora_weights("nerijs/pixel-art-xl", adapter_name="pixel")

# Generate with the LoRA style
image = pipe(
    "A cozy Indian chai stall at sunset, pixel art style",
    num_inference_steps=25,
    guidance_scale=7.5,
    cross_attention_kwargs={"scale": 0.8},  # LoRA strength (0-1)
).images[0]

image.save("chai_pixel_art.png")

# ── Unload LoRA and try another ──
pipe.unload_lora_weights()

print("""
LoRA ecosystem:

  Where to find LoRAs:
  • CivitAI.com — largest collection (100K+ LoRAs)
  • Hugging Face — curated, easy to load
  
  Types of LoRAs:
  • Style LoRAs: pixel art, anime, oil painting, watercolor
  • Subject LoRAs: specific characters, products, faces
  • Concept LoRAs: specific lighting, composition, mood
  
  Key parameter:
  • scale = 0.0 → LoRA has no effect (base model)
  • scale = 0.5 → gentle influence
  • scale = 0.8 → strong influence (recommended)
  • scale = 1.0 → maximum effect (sometimes too strong)
""")


**`part5_lora_train.py`** · _Block 8 of 10_

TRAIN YOUR OWN LoRA — Teach SD to generate YOUR face / product / style


In [ ]:
# =============================================
# TRAIN YOUR OWN LoRA
# Teach SD to generate YOUR face / product / style
# Needs: 20-30 images + 1 GPU + 30 minutes
# =============================================

# The easiest way: use the `autotrain` CLI
# pip install autotrain-advanced

# Step 1: Prepare 20-30 images in a folder
# my_images/
#   img_001.jpg  (512x512, clear, well-lit)
#   img_002.jpg
#   ... (20+ images)

# Step 2: Run training (one command!)
# autotrain dreambooth \
#   --model stabilityai/stable-diffusion-2-1 \
#   --project-name my-style-lora \
#   --image-path ./my_images \
#   --prompt "a photo of sks style" \
#   --resolution 512 \
#   --batch-size 1 \
#   --num-steps 1000 \
#   --lr 1e-4 \
#   --use-lora

# Step 3: Use your trained LoRA
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-2-1", torch_dtype=torch.float16
).to("cuda")

# Load YOUR LoRA
pipe.load_lora_weights("./my-style-lora")

# Generate in YOUR style
image = pipe("A landscape in sks style, beautiful mountains").images[0]
image.save("my_style_landscape.png")

print("""
What just happened:
  - You trained a LoRA (~2-50 MB file) on 20 images
  - It learned your specific style/subject
  - Loading it takes 1 second
  - Base model stays unchanged (860M parameters untouched)
  - You can share the LoRA file — others can use your style!
""")


> **What just happened?** Two things: (1) We loaded a pre-made pixel art LoRA and generated images in that style with one line. (2) We saw how to train your own LoRA on 20 images using autotrain — one CLI command, 30 minutes, produces a tiny file (~10 MB) that teaches SD your style. The base model (860M params) stays untouched. You just add a small adapter on top. LoRA is how creators customize AI image generation for specific products, characters, and art styles.


### Step 6 · ComfyUI — The Professional Workflow Tool

Node-based visual interface for building complex image generation pipelines.

**`part6_comfyui_api.py`** · _Block 9 of 10_

COMFYUI: Use it programmatically via API — ComfyUI runs as a local server. You can send


In [ ]:
# =============================================
# COMFYUI: Use it programmatically via API
# ComfyUI runs as a local server. You can send
# workflow JSON via HTTP to automate it.
# =============================================

# Step 1: Install and start ComfyUI
# git clone https://github.com/comfyanonymous/ComfyUI
# cd ComfyUI
# pip install -r requirements.txt
# python main.py  (starts on http://localhost:8188)

import json
import requests
import io
from PIL import Image

COMFY_URL = "http://localhost:8188"

# ── A simple text-to-image workflow as JSON ──
workflow = {
    "3": {  # KSampler node
        "class_type": "KSampler",
        "inputs": {
            "seed": 42,
            "steps": 25,
            "cfg": 7.5,
            "sampler_name": "euler",
            "scheduler": "normal",
            "denoise": 1.0,
            "model": ["4", 0],
            "positive": ["6", 0],
            "negative": ["7", 0],
            "latent_image": ["5", 0],
        }
    },
    "4": {"class_type": "CheckpointLoaderSimple",
          "inputs": {"ckpt_name": "sd_v2.1.safetensors"}},
    "5": {"class_type": "EmptyLatentImage",
          "inputs": {"width": 512, "height": 512, "batch_size": 1}},
    "6": {"class_type": "CLIPTextEncode",
          "inputs": {"text": "A majestic peacock in an Indian garden, vivid colors", "clip": ["4", 1]}},
    "7": {"class_type": "CLIPTextEncode",
          "inputs": {"text": "blurry, ugly, deformed", "clip": ["4", 1]}},
    "8": {"class_type": "VAEDecode",
          "inputs": {"samples": ["3", 0], "vae": ["4", 2]}},
    "9": {"class_type": "SaveImage",
          "inputs": {"images": ["8", 0], "filename_prefix": "comfy_output"}},
}

# Send to ComfyUI server
def generate_via_comfyui(workflow_json):
    """Send a workflow to ComfyUI and get the result."""
    response = requests.post(
        f"{COMFY_URL}/prompt",
        json={"prompt": workflow_json}
    )
    print(f"Job submitted: {response.json()}")
    return response.json()

# generate_via_comfyui(workflow)  # uncomment when ComfyUI is running

print("""
ComfyUI advantages over pure Python:

  1. VISUAL WORKFLOW: See every node and connection
  2. EASY EXPERIMENTATION: Swap models/LoRAs by clicking
  3. COMPLEX PIPELINES: Chain ControlNet + LoRA + upscaling
  4. REUSABLE WORKFLOWS: Save and share as JSON
  5. COMMUNITY: Thousands of custom nodes on GitHub
  
When to use ComfyUI vs Python:
  • Quick experiment → ComfyUI (visual, fast feedback)
  • Batch processing 1000 images → Python (automation)
  • Building an API endpoint → Python (programmable)
  • Complex multi-model pipeline → ComfyUI (visual debugging)
  • Sharing with non-coders → ComfyUI (no code needed)
""")


> **What just happened?** We saw ComfyUI from two angles: (1) as a visual node editor for building image generation workflows (like connecting LEGO blocks), and (2) as a programmatic API that accepts workflow JSON. The workflow JSON shows exactly how the components connect: checkpoint loader → text encoders → sampler → VAE decoder → save. ComfyUI is what professionals use for complex pipelines (ControlNet + LoRA + upscaling + inpainting in one workflow). Use Python for automation, ComfyUI for experimentation.


### Step 8 · Inference Optimization — 10× Faster Generation

TensorRT, torch.compile, quantization, and few-step models. From 30s to sub-second.

**`optimization.py`** · _Block 10 of 10_

═══════ INFERENCE OPTIMIZATION STACK ═══════


In [ ]:
# ═══════ INFERENCE OPTIMIZATION STACK ═══════

# 1. torch.compile (easiest, 1.5× speedup)
import torch
pipe.unet = torch.compile(pipe.unet, mode="max-autotune", fullgraph=True)
# First call is slow (compilation), subsequent calls 1.5× faster

# 2. TensorRT FP8 (best speedup: 2-2.4×, NVIDIA GPUs only)
# torch.compile(pipe.unet, backend="torch_tensorrt")
# SD 3.5 + TensorRT FP8: 2× faster + 40% less VRAM

# 3. Quantization (lower VRAM, faster inference)
# BF16: default, no quality loss (Ampere+ GPUs)
# FP8:  2× faster, 75% less VRAM (H100/Ada only)
# NF4:  Flux Dev 33GB → 9GB VRAM (slight quality loss)

# 4. Few-step models (BIGGEST single speedup)
# SDXL Lightning: 2-4 steps at full 1024×1024
# LCM-LoRA: plug into any SD/SDXL, 4 steps, guidance=1.5
# Flux Schnell: native 4-step, no distillation needed

# 5. Memory optimization (for constrained GPUs)
pipe.enable_model_cpu_offload()    # Move to CPU between steps
pipe.enable_vae_tiling()           # Process VAE in tiles
pipe.enable_attention_slicing(1)   # Slice attention for low VRAM
# Note: xFormers is deprecated — use PyTorch native SDPA instead

# Combined stack: torch.compile + FP8 + Schnell 4-step
# Result: 30s → sub-1s generation on H100


> **What just happened?** The optimization stack delivers compound speedups. torch.compile (1.5×, one line) → TensorRT FP8 (2.4×, NVIDIA only) → NF4 quantization (Flux 33GB→9GB) → few-step models (SDXL Lightning 4-step, LCM-LoRA). Combined: 30s→sub-1s. xFormers is deprecated — PyTorch native SDPA is faster on modern GPUs. For low VRAM: enable_model_cpu_offload() + vae_tiling(). The RTX 4090 (24GB, ~₹1.3L) is the best consumer GPU — runs SDXL natively and Flux with NF4 quantization.


---

## Wrap-up

You've walked through all 10 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
